# Analysis

Run this notebook **after** `python src/run_eval.py` has produced `results/processed/results.csv`.

It builds the metrics table (accuracy and calibration per model, method and language) and saves a reliability diagram for each language.

In [ ]:
# Make the src modules importable and load the results
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import matplotlib.pyplot as plt
from utils import resolve_path, load_config
import metrics

config = load_config()
df = pd.read_csv(resolve_path(config['paths']['processed_results']))
print(f'{len(df)} rows loaded')
df.head()

## How often could we parse the model's reply?
If `parse_ok` is low for some model, look at a few raw files in `results/raw_outputs/` and adjust the prompt or the parser.

In [ ]:
df.groupby(['model_id', 'method'])['parse_ok'].mean().round(3)

## The main results table
Lower `ece` and `brier` mean better calibration. Positive `overconfidence` means the model is more sure than it is accurate. This is the core table for your paper.

In [ ]:
summary = metrics.summarize(df)
summary

In [ ]:
# Save the table so you can drop it straight into the paper
summary.to_csv(resolve_path('results/processed/summary.csv'), index=False)
print('saved results/processed/summary.csv')

## Reliability diagrams
Pick one model and one method, then draw the curve for each language. The closer the curve sits to the dashed diagonal, the better calibrated the model is in that language.

In [ ]:
model_to_plot = 'gpt-4o'      # change to any model_id in your results
method_to_plot = 'verbalized' # or 'logprob'

figures_dir = resolve_path(config['paths']['figures_dir'])
figures_dir.mkdir(parents=True, exist_ok=True)

clean = df.dropna(subset=['is_correct', 'confidence'])
subset = clean[(clean.model_id == model_to_plot) & (clean.method == method_to_plot)]

for language in subset['language_name'].unique():
    g = subset[subset.language_name == language]
    save_path = figures_dir / f'reliability_{model_to_plot}_{method_to_plot}_{language}.png'
    fig = metrics.reliability_diagram(
        g['confidence'].to_numpy(),
        g['is_correct'].to_numpy(),
        title=f'{model_to_plot} | {method_to_plot} | {language}',
        save_path=save_path,
    )
    plt.show()
    print('saved', save_path)